In [1]:
import sys
print(sys.executable)

/Users/marreejachaak/Desktop/untitled folder/venv/bin/python


## Human-in-the-Loop (HITL) implementation for Microsoft AutoGen's Society of Mind (SoM)
## using a UserProxyAgent for human intervention at critical decision points.

## Features:
##   Inner team with 3 specialized agents (Research, Finance, Pharma)
##   Outer team (Supervisor) that routes tasks to inner team or other teams
##   UserProxyAgent integration to allow human approvals, rejections, context updates,and overrides

In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Callable
import time
import uuid
import pprint

# Try to import AutoGen classes if available. 
try:
    # These names are hypothetical placeholders for the real AutoGen API.
    from autogen import UserProxyAgent as AutoGenUserProxyAgent
    from autogen import Agent as AutoGenAgent
    from autogen import TeamNode as AutoGenTeamNode
    from autogen import SupervisorNode as AutoGenSupervisorNode
    AUTOGEN_AVAILABLE = True
except Exception:
    AUTOGEN_AVAILABLE = False

    # --- Mock implementations (used only when `autogen` is not installed) ---
    class AutoGenAgent:
        def __init__(self, name: str):
            self.name = name

        def run(self, task: Dict[str, Any]) -> Dict[str, Any]:
            # Simple deterministic response for demo purposes.
            return {
                "agent": self.name,
                "decision": f"suggestion from {self.name} for {task.get('topic')}",
                "confidence": 0.7,
                "metadata": {},
            }

    class AutoGenUserProxyAgent:
        """A lightweight UserProxyAgent mock that interacts via console input."""
        def __init__(self, name: str = "UserProxy"):
            self.name = name

        def query_user(self, prompt: str, choices: Optional[List[str]] = None, allow_text: bool = True) -> Dict[str, Any]:
            print("\n== USER PROXY PROMPT ==")
            print(prompt)
            if choices:
                print("Choices:")
                for i, c in enumerate(choices, 1):
                    print(f"  {i}. {c}")
            if allow_text:
                print("(You can also type freeform text to add context or 'override: <new decision>')")
            res = input("Your response: ").strip()

            # parse simple override: 'override: <text>'
            if res.lower().startswith("override:"):
                return {"action": "override", "value": res[len("override:"):].strip()}

            if choices:
                try:
                    idx = int(res) - 1
                    if 0 <= idx < len(choices):
                        return {"action": "choice", "value": choices[idx]}
                except Exception:
                    pass

            # fallback: freeform text
            return {"action": "comment", "value": res}

    class AutoGenTeamNode:
        def __init__(self, name: str, agents: Optional[List[AutoGenAgent]] = None):
            self.name = name
            self.agents = agents or []

        def run(self, task: Dict[str, Any]) -> List[Dict[str, Any]]:
            results = []
            for a in self.agents:
                results.append(a.run(task))
            return results

    class AutoGenSupervisorNode:
        def __init__(self, name: str):
            self.name = name

        def decide(self, task: Dict[str, Any]) -> str:
            # naive routing logic, replace with model-based routing in real system
            topic = task.get("topic","")
            if "drug" in topic.lower() or "pharma" in topic.lower():
                return "pharma_team"
            if "finance" in topic.lower() or "investment" in topic.lower():
                return "finance_team"
            return "inner_team"

# ---------------- Implementation ----------------

class UserProxyAgent:
    """Wrapper around AutoGenUserProxyAgent for consistent interface and helper methods.

    In production this should use the official AutoGen UserProxyAgent which can render a
    UI and listen for user callbacks. Here we emulate the behavior via console IO.
    """
    def __init__(self, name: str = "UserProxy"):
        self._impl = AutoGenUserProxyAgent(name=name)

    def ask_approval(self, summary: str, options: Optional[List[str]] = None) -> Dict[str, Any]:
        prompt = f"Agent recommendations: \n{summary}\n\nDo you approve, reject, or override?"
        return self._impl.query_user(prompt, choices=options or ["Approve", "Reject", "Request more info"], allow_text=True)

    def request_additional_context(self, question: str) -> Dict[str, Any]:
        prompt = f"Request for additional context:\n{question}\nProvide details or constraints (or type 'skip')."
        return self._impl.query_user(prompt, choices=None, allow_text=True)

# Define specialized agents
class ResearchAgent(AutoGenAgent):
    def run(self, task: Dict[str, Any]) -> Dict[str, Any]:
        # A placeholder for a larger LLM reasoning result
        return {
            "agent": self.name,
            "decision": f"Research summary for topic '{task.get('topic')}'",
            "confidence": 0.75,
            "raw": {"sources": ["doc1", "doc2"]},
        }

class FinanceAgent(AutoGenAgent):
    def run(self, task: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "agent": self.name,
            "decision": f"Finance recommendation for '{task.get('topic')}' (e.g., risk low/medium/high)",
            "confidence": 0.65,
        }

class PharmaAgent(AutoGenAgent):
    def run(self, task: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "agent": self.name,
            "decision": f"Pharma insight for '{task.get('topic')}' (safety signals, trial phases)",
            "confidence": 0.8,
        }

# Team wrapper that collects agent outputs and provides a consolidated recommendation
class InnerTeam:
    def __init__(self, name: str, agents: List[AutoGenAgent], user_proxy: UserProxyAgent):
        self.name = name
        self.agents = agents
        self.user_proxy = user_proxy

    def process(self, task: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[InnerTeam:{self.name}] Running {len(self.agents)} agents...\n")
        results = [a.run(task) for a in self.agents]
        # Aggregate results into a single summary
        summary_lines = []
        for r in results:
            summary_lines.append(f"- {r['agent']}: {r['decision']} (conf={r.get('confidence')})")
        summary = "\n".join(summary_lines)

        # Ask user to approve/reject/override
        user_response = self.user_proxy.ask_approval(summary)

        # Interpret user response
        if user_response.get("action") == "override":
            final_decision = user_response.get("value")
            status = "overridden"
        elif user_response.get("action") == "choice":
            choice = user_response.get("value")
            if choice.lower().startswith("approve"):
                final_decision = "Approved: proceed with combined recommendation"
                status = "approved"
            elif choice.lower().startswith("reject"):
                final_decision = "Rejected: do not proceed"
                status = "rejected"
            else:
                final_decision = f"User chose: {choice}"
                status = "user_choice"
        else:
            # comment or other text -> may be additional context
            comment = user_response.get("value")
            if comment and comment.lower() not in ("", "skip"):
                # The user provided extra constraints; ask agents to re-run with constraints.
                print("User provided additional context; re-running agents with constraints...")
                task2 = dict(task)
                task2.setdefault("user_constraints", {})
                task2["user_constraints"]["notes"] = comment
                results2 = [a.run(task2) for a in self.agents]
                summary_lines2 = [f"- {r['agent']}: {r['decision']} (conf={r.get('confidence')})" for r in results2]
                final_decision = "Revised after additional context:\n" + "\n".join(summary_lines2)
                status = "revised"
            else:
                final_decision = "No explicit approval given; defaulting to agent recommendation"
                status = "deferred"

        return {
            "team": self.name,
            "results": results,
            "final_decision": final_decision,
            "status": status,
        }

# Supervisor/outer team that routes tasks
class Supervisor:
    def __init__(self, name: str, teams: Dict[str, InnerTeam], user_proxy: UserProxyAgent):
        self.name = name
        self.teams = teams
        self.user_proxy = user_proxy

    def route_and_execute(self, task: Dict[str, Any]) -> Dict[str, Any]:
        # Simplified routing logic; in real AutoGen SoM this could be a learned router node.
        topic = task.get("topic","")
        # Example: if task contains 'pharma' -> pharma_team; 'finance' -> finance_team; else inner_team
        if "pharma" in topic.lower() or "drug" in topic.lower():
            chosen = "pharma_team"
        elif "finance" in topic.lower() or "investment" in topic.lower():
            chosen = "finance_team"
        else:
            chosen = "inner_team"

        # Let user override routing at critical decision point
        prompt = f"Supervisor routing decision: recommended team '{chosen}'.\nDo you want to accept routing or override to another team?"
        res = self.user_proxy._impl.query_user(prompt, choices=["Accept routing", "Override to finance_team", "Override to pharma_team", "Override to inner_team"], allow_text=True)

        if res.get("action") == "choice":
            val = res.get("value")
            if "override" in val.lower() or "finance" in val.lower() or "pharma" in val.lower() or "inner" in val.lower():
                # naive parse: last token
                chosen = val.split()[-1]
        elif res.get("action") == "override":
            chosen = res.get("value")

        if chosen not in self.teams:
            print(f"Chosen team '{chosen}' not available, falling back to inner_team")
            chosen = "inner_team"

        print(f"Supervisor: routing to {chosen}")
        team = self.teams[chosen]
        return team.process(task)

# ---------------- Demo / Runner ----------------

def build_demo_system() -> Dict[str, Any]:
    user_proxy = UserProxyAgent()

    # Create specialized agents
    research = ResearchAgent("ResearchAgent")
    finance = FinanceAgent("FinanceAgent")
    pharma = PharmaAgent("PharmaAgent")

    # Inner team contains 3 specialized agents
    inner_team = InnerTeam("inner_team", [research, finance, pharma], user_proxy)

    # We create team wrappers for direct team routing as well
    finance_team = InnerTeam("finance_team", [finance], user_proxy)
    pharma_team = InnerTeam("pharma_team", [pharma], user_proxy)

    teams = {
        "inner_team": inner_team,
        "finance_team": finance_team,
        "pharma_team": pharma_team,
    }

    supervisor = Supervisor("supervisor", teams, user_proxy)

    return {
        "user_proxy": user_proxy,
        "teams": teams,
        "supervisor": supervisor,
    }


def demo_interaction():
    system = build_demo_system()
    supervisor: Supervisor = system["supervisor"]

    print("\n=== HITL Society of Mind Demo ===\n")
    while True:
        topic = input("Enter a task topic (or 'quit' to exit): ").strip()
        if topic.lower() in ("quit","exit"):
            print("Exiting demo.")
            break
        task_id = str(uuid.uuid4())
        task = {"id": task_id, "topic": topic, "payload": {}}

        result = supervisor.route_and_execute(task)

        print("\n--- Final Result ---")
        pprint.pprint(result)
        print("--------------------\n")


if __name__ == "__main__":
    demo_interaction()



=== HITL Society of Mind Demo ===


== USER PROXY PROMPT ==
Supervisor routing decision: recommended team 'inner_team'.
Do you want to accept routing or override to another team?
Choices:
  1. Accept routing
  2. Override to finance_team
  3. Override to pharma_team
  4. Override to inner_team
(You can also type freeform text to add context or 'override: <new decision>')
Supervisor: routing to inner_team

[InnerTeam:inner_team] Running 3 agents...


== USER PROXY PROMPT ==
Agent recommendations: 
- ResearchAgent: Research summary for topic 'Impact of AI in Healthcare' (conf=0.75)
- FinanceAgent: Finance recommendation for 'Impact of AI in Healthcare' (e.g., risk low/medium/high) (conf=0.65)
- PharmaAgent: Pharma insight for 'Impact of AI in Healthcare' (safety signals, trial phases) (conf=0.8)

Do you approve, reject, or override?
Choices:
  1. Approve
  2. Reject
  3. Request more info
(You can also type freeform text to add context or 'override: <new decision>')
User provided additi